<a href="https://colab.research.google.com/github/4xArjun/my-to-do-app/blob/main/flood__Ready_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np

print("--- FLOODREADY: 48-HOUR FLOOD PREDICTION & RELIEF SYSTEM ---\n")

# 1. Messy Sensor Data & Village Data
rivers_data = {
    'river_id': ['R01', 'R02', 'R03', 'R04'],
    'river_name': ['Ganga Sector A', 'Kaveri Basin B', 'Periyar Sector C', 'Narmada Reach D'],
    'current_level_m': [4.5, 6.2, 3.8, 5.9],
    'danger_threshold_m': [5.0, 6.0, 5.0, 6.5],
    'rain_forecast_48h_mm': [80.0, 120.0, 15.0, 95.0],
    'runoff_coeff': [0.015, 0.020, 0.010, 0.018]
}

villages_data = {
    'village_id': ['V01', 'V02', 'V03', 'V04'],
    'village_name': ['Rampur', 'Kalyanpur', 'Sunderpur', 'Anandgram'],
    'assigned_river': ['R01', 'R02', 'R03', 'R04'],
    'population': [3200, 4500, 1200, 2800],
    'vulnerability_score': [0.8, 0.9, 0.4, 0.7] # Scale 0 to 1
}

df_rivers = pd.DataFrame(rivers_data)
df_villages = pd.DataFrame(villages_data)





# 2. Predict 48-Hour River Levels
df_rivers['predicted_level_48h'] = (
    df_rivers['current_level_m'] + (df_rivers['rain_forecast_48h_mm'] * df_rivers['runoff_coeff'])
)
df_rivers['will_flood_48h'] = df_rivers['predicted_level_48h'] >= df_rivers['danger_threshold_m']




# 3. Merge & Prioritize Villages for Relief
merged = df_villages.merge(df_rivers, left_on='assigned_river', right_on='river_id')

# Priority Score = Flood Status (1 or 0) * Vulnerability * Population
merged['priority_score'] = (
    merged['will_flood_48h'].astype(int) * merged['vulnerability_score'] * merged['population']
)

priority_list = merged.sort_values(by='priority_score', ascending=False)






# 4. Generate Readable Rescue Alerts
def generate_alert(row):
    if row['will_flood_48h']:
        return (f"CRITICAL RED ALERT: {row['village_name']} near {row['river_name']} will flood in 48h! "
                f"Predicted level: {row['predicted_level_48h']:.2f}m (Threshold: {row['danger_threshold_m']}m). "
                f"IMMEDIATE EVACUATION REQUIRED FOR {row['population']} RESIDENTS.")
    return f"STATUS NORMAL: {row['village_name']} water levels stable over next 48h."

priority_list['dispatch_alert'] = priority_list.apply(generate_alert, axis=1)




#5. Display Results
for idx, row in priority_list.iterrows():
    print(f"Priority Score: {row['priority_score']:.1f}")
    print(f"Alert: {row['dispatch_alert']}\n")

--- FLOODREADY: 48-HOUR FLOOD PREDICTION & RELIEF SYSTEM ---

Priority Score: 4050.0
Alert: CRITICAL RED ALERT: Kalyanpur near Kaveri Basin B will flood in 48h! Predicted level: 8.60m (Threshold: 6.0m). IMMEDIATE EVACUATION REQUIRED FOR 4500 RESIDENTS.

Priority Score: 2560.0
Alert: CRITICAL RED ALERT: Rampur near Ganga Sector A will flood in 48h! Predicted level: 5.70m (Threshold: 5.0m). IMMEDIATE EVACUATION REQUIRED FOR 3200 RESIDENTS.

Priority Score: 1960.0
Alert: CRITICAL RED ALERT: Anandgram near Narmada Reach D will flood in 48h! Predicted level: 7.61m (Threshold: 6.5m). IMMEDIATE EVACUATION REQUIRED FOR 2800 RESIDENTS.

Priority Score: 0.0
Alert: STATUS NORMAL: Sunderpur water levels stable over next 48h.

